# Continuation pack (play-a-hand) — headless, on Kaggle

Re-solves the **continuation** content (linked flop→turn→river hands where the villain
plays its solved main line) and writes the signed `continuation_seed` pack.

**Why no GPU here.** Every other content notebook uses the GPU solver (`BatchedGPUCFR`).
Continuation is the exception: it runs on `MultiStreetSpike`, which is NumPy/CPU-only, and
the trajectory-extraction API it needs (`eval_capture_targets` / `decision` /
`node_action_share`) lives only there. So a GPU runtime won't speed this up — pick a plain
**CPU** session. This just offloads the long batch to Kaggle instead of your Mac.

**Runtime.** ~30–35 min per flop on Kaggle's CPU. The 4 curated boards ≈ 2–2.5 h, well within
a Kaggle CPU session. Needs **Internet On** (Settings → to clone the repo).

This run includes the `ovb` reach-filter fix (commit `f8f3928`): the villain now folds to
hero bets where the solver does, instead of always calling.

In [ ]:
# Clone the solver source (needs Internet On). Pulls the ovb fix.
!rm -rf /kaggle/working/poker && git clone -q --depth 1 https://github.com/tian-chaiyaporn2/poker_offline_trainer /kaggle/working/poker
import sys; sys.path.insert(0, '/kaggle/working/poker/src')
import subprocess
print('source ready @', subprocess.run(['git','-C','/kaggle/working/poker','rev-parse','--short','HEAD'],
                                       capture_output=True, text=True).stdout.strip())

In [ ]:
# Solve the continuation library on CPU and write the signed pack.
#   FLOPS  : how many curated (flop,turn,river) runouts to solve. Max 4 today; to add more,
#            append tuples to CURATED in demo/gen_continuation.py (line ~42), then raise FLOPS.
#   N      : combos sampled per range (richer than the 40-combo seed).
#   ITERS  : CFR iterations. The run prints stable=True/False per flop — you want all True;
#            if any is False, bump ITERS (e.g. 600) and re-run rather than shipping it.
#   VERSION: keep 'continuation_seed' to drop-in replace the shipped pack.
import subprocess, os
FLOPS, N, ITERS, VERSION = 4, 60, 400, 'continuation_seed'
env = {**os.environ, 'PYTHONPATH': 'src'}
subprocess.run(
    ['python', 'demo/gen_continuation.py',
     '--flops', str(FLOPS), '--n', str(N), '--iters', str(ITERS), '--version', VERSION],
    cwd='/kaggle/working/poker', env=env, check=True)

In [ ]:
# Expose the three pack files for download (right-click → Download in the Output panel).
import shutil, os
VERSION = 'continuation_seed'
base = '/kaggle/working/poker/output/packs'
files = [f'flop_pack_{VERSION}.db', f'flop_pack_{VERSION}.db.gz', f'build_report_{VERSION}.json']
src = os.path.join(base, files[0])
if not os.path.exists(src):
    raise SystemExit('No pack written -- check the solve cell above for stable=False / errors.')
for f in files:
    shutil.copy(os.path.join(base, f), os.path.join('/kaggle/working', f))
print('DOWNLOAD these from /kaggle/working:')
for f in files:
    print('  %-42s %d KB' % (f, os.path.getsize(os.path.join('/kaggle/working', f)) // 1024))